In [1]:
import os
import json
import duckdb
import hashlib

# Setup paths
FHIR_DIR = "../synthea/output/fhir"
DB_PATH = "../data/omop_clinical.duckdb"

def stable_person_id(source_id: str) -> int:
    """Generates a deterministic person_id using SHA-256 for perfect relational integrity."""
    return int(hashlib.sha256(source_id.encode()).hexdigest(), 16) % (10**9)

def extract_drugs(patient_file):
    """
    Reads a FHIR bundle and extracts MedicationRequest records.
    Returns a list of tuples ready for bulk database insertion.
    """
    file_path = os.path.join(FHIR_DIR, patient_file)
    with open(file_path, 'r', encoding='utf-8') as f:
        fhir_data = json.load(f)
        
    drugs = []
    
    for entry in fhir_data.get('entry', []):
        resource = entry.get('resource', {})
        
        # We only want Medication resources (prescriptions)
        if resource.get('resourceType') == 'MedicationRequest':
            
            # Extract Patient Reference and hash it deterministically
            subject_ref = resource.get('subject', {}).get('reference', '')
            patient_source_id = subject_ref.replace('urn:uuid:', '')
            person_id = stable_person_id(patient_source_id)
            
            # Extract RxNorm Code and Text Description
            medication = resource.get('medicationCodeableConcept', {})
            coding = medication.get('coding', [])
            rxnorm_code = "0"
            drug_text = "Unknown"
            
            if coding:
                # Synthea uses RxNorm codes for medications
                rxnorm_code = coding[0].get('code', '0')
                drug_text = coding[0].get('display', 'Unknown')
                
            # Extract Dates (authoredOn is the prescription date)
            start_date = resource.get('authoredOn', '1900-01-01')[:10] 
            
            drugs.append((
                person_id,
                rxnorm_code,
                drug_text,
                start_date
            ))
            
    return drugs

print("⚙️ STARTING ETL PIPELINE (FHIR -> OMOP DRUG_EXPOSURE)\n" + "-"*50)

# 1. Extraction Phase (Python)
print("🔍 Extracting medications from FHIR JSON files...")
json_files = [f for f in os.listdir(FHIR_DIR) if f.endswith('.json')]
all_drugs = []

for file in json_files:
    all_drugs.extend(extract_drugs(file))

print(f"📊 Extracted {len(all_drugs)} raw medication records.")

# 2. Load & Transform Phase (DuckDB)
print("🔌 Connecting to DuckDB for RxNorm vocabulary mapping...")

try:
    with duckdb.connect(DB_PATH) as con:
        
        # Staging table
        con.execute("DROP TABLE IF EXISTS stg_drug")
        con.execute("""
            CREATE TEMPORARY TABLE stg_drug (
                person_id BIGINT,
                rxnorm_code VARCHAR,
                drug_text VARCHAR,
                start_date DATE
            )
        """)
        
        con.executemany("""
            INSERT INTO stg_drug VALUES (?, ?, ?, ?)
        """, all_drugs)
        
        print("⏳ Preparing standard OMOP DRUG_EXPOSURE table (Idempotent)...")
        # Idempotency step: create if not exists, then clear to prevent duplicates on re-runs
        con.execute("""
            CREATE TABLE IF NOT EXISTS drug_exposure (
                drug_exposure_id BIGINT PRIMARY KEY,
                person_id BIGINT,
                drug_concept_id INTEGER,
                drug_exposure_start_date DATE,
                drug_source_value VARCHAR,
                drug_source_concept_id INTEGER
            )
        """)
        con.execute("DELETE FROM drug_exposure")
        
        print("🧠 Performing SQL JOIN with OMOP Concept table to translate RxNorm codes...")
        con.execute("""
            INSERT INTO drug_exposure 
            SELECT 
                ROW_NUMBER() OVER () AS drug_exposure_id,
                stg.person_id,
                COALESCE(c.concept_id, 0) AS drug_concept_id, 
                stg.start_date AS drug_exposure_start_date,
                stg.drug_text AS drug_source_value,
                COALESCE(c.concept_id, 0) AS drug_source_concept_id -- Strict OMOP compliance: Integer Concept ID
            FROM stg_drug stg
            LEFT JOIN concept c 
                ON stg.rxnorm_code = c.concept_code 
                AND c.vocabulary_id = 'RxNorm'
                AND c.domain_id = 'Drug'
        """)
        
        # Verification
        mapped_count = con.execute("SELECT COUNT(*) FROM drug_exposure WHERE drug_concept_id != 0").fetchone()[0]
        unmapped_count = con.execute("SELECT COUNT(*) FROM drug_exposure WHERE drug_concept_id = 0").fetchone()[0]
        
        print(f"\n✅ ETL Complete!")
        print(f" - Successfully mapped to OMOP Standards: {mapped_count} medications")
        print(f" - Failed to map (Unknown/Custom): {unmapped_count} medications")
        
        print("\n🔎 Sample of standard mapped medications:")
        sample = con.execute("""
            SELECT person_id, drug_concept_id, drug_source_value, drug_exposure_start_date 
            FROM drug_exposure 
            WHERE drug_concept_id != 0
            LIMIT 5
        """).fetchall()
        
        for row in sample:
            print(f" - Person: {row[0]:<15} | Concept ID: {row[1]:<10} | Date: {row[3]} | Drug: {row[2]}")

except Exception as e:
    print(f"❌ Database error: {e}")

⚙️ STARTING ETL PIPELINE (FHIR -> OMOP DRUG_EXPOSURE)
--------------------------------------------------
🔍 Extracting medications from FHIR JSON files...
📊 Extracted 1461 raw medication records.
🔌 Connecting to DuckDB for RxNorm vocabulary mapping...
⏳ Preparing standard OMOP DRUG_EXPOSURE table (Idempotent)...
🧠 Performing SQL JOIN with OMOP Concept table to translate RxNorm codes...

✅ ETL Complete!
 - Successfully mapped to OMOP Standards: 1016 medications
 - Failed to map (Unknown/Custom): 445 medications

🔎 Sample of standard mapped medications:
 - Person: 398615133       | Concept ID: 19073094   | Date: 2026-05-22 | Drug: amLODIPine 2.5 MG Oral Tablet
 - Person: 231967926       | Concept ID: 19073183   | Date: 2024-04-03 | Drug: Amoxicillin 250 MG Oral Capsule
 - Person: 29670787        | Concept ID: 19073188   | Date: 2026-04-13 | Drug: Amoxicillin 500 MG Oral Tablet
 - Person: 398615133       | Concept ID: 19073094   | Date: 2025-05-16 | Drug: amLODIPine 2.5 MG Oral Tablet
 - P